[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/B.%20Core%20Quay-Side%20Operations%20%28The%20Ship-to-Shore%20Interface%29/12.%20The%20Berth%20Allocation%20with%20Tidal%20Windows%20Problem/P12-Tier-7_executed.ipynb)

---


# 12. The Berth Allocation with Tidal Windows Problem
## Tier 7 — The Human–AI Symbiotic Partnership (The Centaur Model)

### Goal
This notebook demonstrates a human-in-the-loop decision process for a berth-allocation scenario. A transparent, priority-ordered AI heuristic produces an initial schedule; a human expert reviews it, can suggest a change, and the system checks and evaluates the resulting schedule.

The purpose is to study the roles of explanation, contextual judgment, feedback, and trust in a collaborative workflow. The heuristic is a teaching model, not a general-purpose or globally optimal berth-allocation solver.

### Inputs and outputs
- **Inputs:** vessel arrival times, drafts, lengths, processing durations, demurrage rates, berth dimensions and depths, tidal access windows, and a human expert profile.
- **Outputs:** an AI schedule, human feedback, a refined schedule, and diagnostic collaboration measures. The notebook also plots and compares the AI and collaborative schedules.

### Scope and modeling assumptions
- Time is measured in hours on a single, unwrapped timeline. The tidal window from 23:00 to 02:00 is represented as 23 to 26, so time does not jump backward at midnight.
- A tidal window limits the time at which a vessel may start its berth entry. The vessel must meet the berth-depth requirement at that start time; its processing interval may extend beyond the tidal window.
- Each berth can serve at most one vessel at a time. Processing time is the berth-occupancy duration, and vessels assigned to a berth are scheduled without overlap.
- The listed tugboat and pilot resources provide operational context, but the scenario does not specify movement durations or resource-use intervals. They are therefore not enforced as scheduling constraints.
- Weather, channel traffic, safety margins, uncertainty, and real-time human-interface behavior are outside this illustrative model. The feedback and trust measures are heuristic indicators, not validated psychological or operational measurements.

### Method and learning sequence
The AI sorts vessels by commercial priority and then demurrage rate, and greedily assigns each vessel to the feasible berth and start time with the lowest modeled cost. The human then reviews that recommendation. A proposed berth reassignment is accepted only when the berth can accommodate the vessel and a feasible, non-overlapping start time can be found. The notebook finally reports diagnostic metrics and limitations; it does not claim that the heuristic finds a global optimum or that one simulated interaction proves a performance improvement.

In [ ]:
# These libraries support the scenario data, heuristic workflow, reporting, and plots.
import copy
import random
import time
from collections import defaultdict
from dataclasses import dataclass
from typing import Dict, List, Optional

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

# The fixed seeds make any future use of these random-number generators reproducible.
np.random.seed(42)
random.seed(42)

print("Libraries imported successfully.")

In [ ]:
@dataclass
class Vessel:
    """Vessel data, including preference and human-context attributes."""

    id: int
    name: str
    draft: float
    length: float
    processing_time: float
    arrival_time: float
    demurrage_cost: float
    captain_preference: Optional[int] = None
    captain_relationship: float = 0.5
    commercial_priority: float = 0.5
    flexibility_score: float = 0.5


@dataclass
class HumanExpert:
    """Expertise and preference information used in the human review."""

    id: int
    name: str
    experience_years: int
    expertise_areas: List[str]
    risk_tolerance: float
    decision_style: str
    trust_in_ai: float
    vessel_relationships: Dict[int, float]


@dataclass
class AIRecommendation:
    """AI schedule and accompanying explanatory information."""

    vessel_assignments: Dict[int, int]
    start_times: Dict[int, float]
    objective_value: float
    confidence_score: float
    explanation: str
    decision_factors: List[Dict]
    alternatives: List[Dict]


@dataclass
class HumanFeedback:
    """Human review, proposed changes, and confidence in that review."""

    expert_id: int
    recommendation_id: str
    acceptance_score: float
    modifications: Dict
    reasoning: str
    confidence_in_decision: float


@dataclass
class CollaborativeSolution:
    """Schedule and diagnostic measures after human review."""

    vessel_assignments: Dict[int, int]
    start_times: Dict[int, float]
    objective_value: float
    human_satisfaction_score: float
    ai_alignment_score: float
    trust_evolution: float
    collaboration_quality: float
    final_confidence: float

## 1. Initialize the Rotterdam scenario

A berth is feasible only if its length is at least the vessel's length and its available depth at entry is at least the vessel's draft. The depth within a tidal window is interpolated between the berth's low and high depths using the window's `depth_factor`. The scenario also records human preferences and resource availability; only constraints that can be evaluated from the supplied data are enforced by the schedule heuristic.

In [ ]:
def initialize_rotterdam_scenario():
    """Create the vessel, berth, tidal-window, and expert data."""

    berths = [
        {"id": 0, "name": "Berth A", "length": 400, "depth_low": 12.0, "depth_high": 16.5},
        {"id": 1, "name": "Berth B", "length": 350, "depth_low": 11.0, "depth_high": 15.5},
        {"id": 2, "name": "Berth C", "length": 450, "depth_low": 13.5, "depth_high": 18.0},
        {"id": 3, "name": "Berth D", "length": 380, "depth_low": 10.5, "depth_high": 15.0},
    ]

    vessels = [
        Vessel(0, "MSC_Mediterranean", 14.2, 350, 6.5, 14.0, 12000, 1, 0.8, 0.9, 0.3),
        Vessel(1, "COSCO_Shanghai", 13.8, 380, 7.0, 15.5, 11000, None, 0.6, 0.7, 0.6),
        Vessel(2, "Maersk_Edinburgh", 15.1, 400, 8.0, 16.0, 15000, 2, 0.9, 0.8, 0.4),
        Vessel(3, "ONE_Efficiency", 12.5, 320, 5.5, 17.0, 9000, 0, 0.5, 0.6, 0.8),
        Vessel(4, "Hapag_Lloyd", 13.2, 360, 6.0, 18.0, 10000, None, 0.7, 0.5, 0.7),
    ]

    human_expert = HumanExpert(
        id=1,
        name="Captain van der Meer",
        experience_years=25,
        expertise_areas=["tidal", "commercial", "operations"],
        risk_tolerance=0.4,
        decision_style="collaborative",
        trust_in_ai=0.7,
        vessel_relationships={0: 0.8, 1: 0.6, 2: 0.9, 3: 0.5, 4: 0.7},
    )

    # The end value 26 represents 02:00 on the following day.
    tidal_windows = [
        {"start": 13.0, "end": 16.0, "depth_factor": 0.8},
        {"start": 19.0, "end": 22.0, "depth_factor": 1.0},
        {"start": 23.0, "end": 26.0, "depth_factor": 0.9},
    ]

    resource_constraints = {
        "tugboats_available": 2,
        "pilots_available": 2,
        "stevedores_available": 15,
        "equipment_status": {
            "tugboat_1": True,
            "tugboat_2": True,
            "tugboat_3": False,
            "pilot_boat_1": True,
            "pilot_boat_2": True,
        },
    }

    return {
        "vessels": vessels,
        "berths": berths,
        "human_expert": human_expert,
        "tidal_windows": tidal_windows,
        "resource_constraints": resource_constraints,
    }


scenario = initialize_rotterdam_scenario()
print("Rotterdam scenario initialized:")
print(f"  Vessels: {len(scenario['vessels'])}")
print(f"  Berths: {len(scenario['berths'])}")
print(f"  Tidal access windows: {len(scenario['tidal_windows'])}")
print(f"  Tugboats listed as available: {scenario['resource_constraints']['tugboats_available']}")
print(f"  Human expert: {scenario['human_expert'].name} ({scenario['human_expert'].experience_years} years of experience)")

## 2. AI recommendation: a greedy, priority-ordered heuristic

### Intuition and formal structure
The heuristic considers one vessel at a time in priority order. For each vessel, it examines berths and tidal windows, selecting the feasible option with the lowest modeled cost. A berth's existing schedule matters: the next vessel assigned there cannot start until the previous vessel's processing interval ends.

Let $V$ be the vessel set, $B$ the berth set, and $W$ the set of tidal windows. For vessel $i$, berth $b$, and candidate start time $t$, define $d_i$ as the arrival time and $p_i$ as the processing duration. The cost used by the heuristic is

$$c_{ib}(t) = r_i\max(0, t-d_i)\,q_{ib}, \qquad
q_{ib}=\begin{cases}0.9,&\text{if berth }b\text{ is vessel }i\text{'s stated preference},\\1,&\text{otherwise.}\end{cases}$$

Here $r_i$ is the vessel's demurrage rate. The preference factor is a modeled 10% reduction in this cost proxy, not a measured financial saving. A candidate is feasible when the vessel fits the berth, the draft is no greater than the depth available in the selected tidal window, $t$ lies in that window and is no earlier than arrival or berth availability, and the berth is not occupied during $[t,t+p_i)$.

If $S$ is the set of vessels already scheduled, the priority-ordered greedy step is

$$ (b_i,t_i) \in \arg\min_{b\in B,\,t\in F_i(b;S)} c_{ib}(t), $$

where $F_i(b;S)$ is the set of feasible start times for vessel $i$ at berth $b$ given the current partial schedule. If this set is empty for every berth, the vessel remains unassigned. This is a sequential heuristic, not an exact global optimization method. Its simple implementation checks berths and windows for each vessel and tests previously scheduled intervals for berth conflicts; its running time grows with the number of vessels, berths, windows, and already scheduled vessels.

The implementation separates feasibility (`_find_feasible_start_time`), cost calculation (`_vessel_cost`), and the priority-ordered assignment (`_optimize_berth_allocation`) so each decision can be inspected.

In [ ]:
class ExplainableAIOptimizer:
    """Priority-ordered greedy scheduler with explanatory outputs."""

    def __init__(self, scenario_data: Dict):
        self.scenario = scenario_data
        self.vessels = scenario_data["vessels"]
        self.berths = scenario_data["berths"]
        self.tidal_windows = scenario_data["tidal_windows"]
        self.resource_constraints = scenario_data["resource_constraints"]

    def generate_recommendation(self, human_context: Optional[Dict] = None) -> AIRecommendation:
        """Build the AI schedule and attach explanations and diagnostics."""
        solution = self._optimize_berth_allocation()
        explanation = self._generate_explanation(solution, human_context)
        decision_factors = self._analyze_decision_factors(solution)
        alternatives = self._generate_alternatives(solution)
        confidence = self._calculate_confidence(solution, human_context)

        return AIRecommendation(
            vessel_assignments=solution["assignments"],
            start_times=solution["start_times"],
            objective_value=solution["objective"],
            confidence_score=confidence,
            explanation=explanation,
            decision_factors=decision_factors,
            alternatives=alternatives,
        )

    def _vessel_cost(self, vessel: Vessel, berth: Dict, start_time: float) -> float:
        """Return the preference-adjusted demurrage cost proxy."""
        delay = max(0.0, start_time - vessel.arrival_time)
        preference_factor = 0.9 if vessel.captain_preference == berth["id"] else 1.0
        return vessel.demurrage_cost * delay * preference_factor

    def _optimize_berth_allocation(self) -> Dict:
        """Greedily assign vessels in descending priority order."""
        sorted_vessels = sorted(
            self.vessels,
            key=lambda vessel: (vessel.commercial_priority, vessel.demurrage_cost),
            reverse=True,
        )
        assignments = {}
        start_times = {}
        total_cost = 0.0

        for vessel in sorted_vessels:
            best_choice = None
            for berth in self.berths:
                feasible_start = self._find_feasible_start_time(
                    vessel, berth, assignments, start_times
                )
                if feasible_start is None:
                    continue

                cost = self._vessel_cost(vessel, berth, feasible_start)
                if best_choice is None or cost < best_choice[0]:
                    best_choice = (cost, berth["id"], feasible_start)

            if best_choice is not None:
                cost, berth_id, start_time = best_choice
                assignments[vessel.id] = berth_id
                start_times[vessel.id] = start_time
                total_cost += cost

        return {
            "assignments": assignments,
            "start_times": start_times,
            "objective": total_cost,
        }

    def _find_feasible_start_time(
        self,
        vessel: Vessel,
        berth: Dict,
        assignments: Optional[Dict] = None,
        start_times: Optional[Dict] = None,
        exclude_vessel_id: Optional[int] = None,
    ) -> Optional[float]:
        """Find the earliest feasible tidal entry after berth availability."""
        if berth["length"] < vessel.length:
            return None

        assignments = assignments or {}
        start_times = start_times or {}
        occupied_until = []
        for assigned_id, assigned_berth in assignments.items():
            if assigned_id == exclude_vessel_id or assigned_berth != berth["id"]:
                continue
            assigned_vessel = next(item for item in self.vessels if item.id == assigned_id)
            occupied_until.append(
                start_times[assigned_id] + assigned_vessel.processing_time
            )
        berth_available = max(occupied_until, default=float("-inf"))

        for window in sorted(self.tidal_windows, key=lambda item: item["start"]):
            candidate = max(vessel.arrival_time, window["start"], berth_available)
            if candidate > window["end"]:
                continue

            available_depth = berth["depth_low"] + (
                berth["depth_high"] - berth["depth_low"]
            ) * window["depth_factor"]
            if available_depth >= vessel.draft:
                return candidate

        return None

    def _generate_explanation(self, solution: Dict, human_context: Optional[Dict]) -> str:
        """Summarize the actual modeled rules and schedule coverage."""
        assigned = len(solution["assignments"])
        total = len(self.vessels)
        context_note = (
            "Human context was supplied for review."
            if human_context
            else "No additional human context was supplied to the AI step."
        )
        return (
            f"The priority-ordered heuristic assigned {assigned} of {total} vessels. "
            "It checked berth length, draft against depth at tidal entry, arrival time, "
            "and non-overlapping berth occupancy. Its cost proxy combines waiting-time "
            "demurrage with the stated preferred-berth factor. Tugboat and pilot "
            "availability are recorded but are not scheduling constraints in this model. "
            f"{context_note} The result is a heuristic recommendation, not a proof of global optimality."
        )

    def _analyze_decision_factors(self, solution: Dict) -> List[Dict]:
        """List the decision factors represented in this teaching model."""
        return [
            {
                "factor": "Tidal access and draft feasibility",
                "impact": "Hard constraint",
                "weight": 0.35,
                "explanation": "The vessel draft must not exceed depth at its entry time.",
            },
            {
                "factor": "Berth length and occupancy",
                "impact": "Hard constraint",
                "weight": 0.25,
                "explanation": "The vessel must fit, and berth service intervals cannot overlap.",
            },
            {
                "factor": "Commercial priority and demurrage",
                "impact": "Scheduling order and cost proxy",
                "weight": 0.20,
                "explanation": "Priority orders the greedy assignments; delay contributes to cost.",
            },
            {
                "factor": "Captain berth preference",
                "impact": "Cost adjustment",
                "weight": 0.15,
                "explanation": "A stated berth preference applies the modeled cost factor.",
            },
            {
                "factor": "Unmodeled operational context",
                "impact": "Human review",
                "weight": 0.05,
                "explanation": "Weather and resource-use intervals require additional data.",
            },
        ]

    def _generate_alternatives(self, solution: Dict) -> List[Dict]:
        """Describe review perspectives without presenting fabricated schedules."""
        return [
            {
                "name": "Cost-focused review",
                "description": "Review waiting costs and the priority order.",
                "trade_offs": "A change may affect preferences or other vessels' delays.",
                "when_to_use": "When delay cost is the central operational concern.",
            },
            {
                "name": "Relationship-focused review",
                "description": "Review captain preferences and vessel relationships.",
                "trade_offs": "A preferred berth may increase waiting cost or affect feasibility.",
                "when_to_use": "When customer context is important to the decision.",
            },
            {
                "name": "Risk-focused review",
                "description": "Review missing safety, weather, and resource information.",
                "trade_offs": "This notebook cannot quantify risks without additional data.",
                "when_to_use": "When the available schedule data do not capture operational risk.",
            },
        ]

    def _calculate_confidence(self, solution: Dict, human_context: Optional[Dict]) -> float:
        """Return a bounded heuristic confidence indicator, not a probability."""
        completeness = len(solution["assignments"]) / len(self.vessels)
        context_factor = 0.9 if human_context else 0.8
        return min(0.95, max(0.5, 0.85 * completeness * context_factor))


print("Explainable AI heuristic defined.")

## 3. Human review and feedback

The human decision-maker checks preference alignment and delay for assigned vessels, records contextual factors absent from the data, and may propose a specific change. Acceptance, confidence, and trust updates are simple bounded indicators designed for this demonstration; they should not be interpreted as empirical measurements. The scenario is passed explicitly to the decision-maker so that the analysis uses the same vessel data as the AI.

In [ ]:
class HumanDecisionMaker:
    """Represent an expert reviewing an AI recommendation."""

    def __init__(self, human_expert: HumanExpert, scenario_data: Dict):
        self.expert = human_expert
        self.scenario = scenario_data
        self.decision_history = []
        self.trust_evolution = []

    def evaluate_ai_recommendation(self, recommendation: AIRecommendation) -> HumanFeedback:
        """Assess the recommendation, record proposed changes, and update trust."""
        initial = self._initial_assessment(recommendation)
        analysis = self._analyze_recommendation(recommendation)
        modifications = self._identify_modifications(recommendation, analysis)
        acceptance = self._calculate_acceptance_score(initial, analysis, modifications)
        reasoning = self._generate_reasoning(analysis, modifications)
        confidence = self._calculate_decision_confidence(analysis, modifications)

        feedback = HumanFeedback(
            expert_id=self.expert.id,
            recommendation_id=f"rec_{len(self.decision_history) + 1}",
            acceptance_score=acceptance,
            modifications=modifications,
            reasoning=reasoning,
            confidence_in_decision=confidence,
        )
        self._update_trust(recommendation, feedback)
        self.decision_history.append(feedback)
        return feedback

    def _initial_assessment(self, recommendation: AIRecommendation) -> float:
        """Combine initial trust with a small confidence-based adjustment."""
        acceptance = self.expert.trust_in_ai
        acceptance += (recommendation.confidence_score - 0.7) * 0.3
        if self.expert.risk_tolerance < 0.5 and recommendation.confidence_score > 0.8:
            acceptance *= 0.9
        return max(0.1, min(0.9, acceptance))

    def _analyze_recommendation(self, recommendation: AIRecommendation) -> Dict:
        """Check preference matches and delays for assigned vessels."""
        analysis = {
            "strengths": [],
            "concerns": [],
            "missing_factors": [
                "Current weather and visibility",
                "Channel traffic and external congestion",
                "Resource-use intervals for tugboats and pilots",
                "Current vessel and crew circumstances",
            ],
            "relationship_impact": {},
        }

        vessels_by_id = {vessel.id: vessel for vessel in self.scenario["vessels"]}
        for vessel_id, berth_id in recommendation.vessel_assignments.items():
            vessel = vessels_by_id[vessel_id]
            if vessel.captain_preference == berth_id:
                analysis["strengths"].append(
                    f"{vessel.name} is assigned to its preferred berth ({berth_id})."
                )
                analysis["relationship_impact"][vessel_id] = "positive"
            elif vessel.captain_preference is not None:
                analysis["concerns"].append(
                    f"{vessel.name} is not assigned to its preferred berth ({vessel.captain_preference})."
                )
                analysis["relationship_impact"][vessel_id] = "needs review"

            delay = recommendation.start_times[vessel_id] - vessel.arrival_time
            if vessel.commercial_priority > 0.7 and delay < 2:
                analysis["strengths"].append(
                    f"High-priority vessel {vessel.name} has less than two hours of delay."
                )
            elif vessel.commercial_priority > 0.7:
                analysis["concerns"].append(
                    f"High-priority vessel {vessel.name} has at least two hours of delay."
                )
        return analysis

    def _identify_modifications(self, recommendation: AIRecommendation, analysis: Dict) -> Dict:
        """Propose the stated MSC preference when relationship context supports review."""
        modifications = {}
        vessel = next(item for item in self.scenario["vessels"] if item.id == 0)
        current_berth = recommendation.vessel_assignments.get(vessel.id)
        relationship = self.expert.vessel_relationships.get(vessel.id, 0.5)

        if (
            current_berth is not None
            and current_berth != vessel.captain_preference
            and vessel.captain_preference is not None
            and relationship > 0.7
        ):
            modifications[vessel.id] = {
                "type": "berth_reassignment",
                "from": current_berth,
                "to": vessel.captain_preference,
                "reasoning": (
                    "The captain strongly prefers Berth B for easier maneuvering; "
                    "the berth and schedule must still pass feasibility checks."
                ),
            }

        for vessel_id, start_time in recommendation.start_times.items():
            vessel = next(item for item in self.scenario["vessels"] if item.id == vessel_id)
            delay = start_time - vessel.arrival_time
            if vessel.commercial_priority > 0.8 and delay > 3:
                modifications.setdefault(
                    vessel_id,
                    {
                        "type": "priority_review",
                        "current_delay": delay,
                        "reasoning": (
                            f"Review the {delay:.1f}-hour delay for high-priority vessel {vessel.name}."
                        ),
                    },
                )
        return modifications

    def _calculate_acceptance_score(self, initial: float, analysis: Dict, modifications: Dict) -> float:
        """Adjust initial acceptance using simple strength and concern counts."""
        score = initial + 0.05 * len(analysis["strengths"])
        score -= 0.08 * len(analysis["concerns"])
        if modifications:
            score += 0.05 - 0.1 * len(modifications)
        return max(0.1, min(0.9, score))

    def _generate_reasoning(self, analysis: Dict, modifications: Dict) -> str:
        """Create a readable summary of the review and its information gaps."""
        lines = ["The recommendation has been reviewed.", "", "Strengths:"]
        lines.extend(f"- {item}" for item in analysis["strengths"][:3])
        if not analysis["strengths"]:
            lines.append("- No listed strengths were identified by the review rules.")
        lines.extend(["", "Concerns:"])
        lines.extend(f"- {item}" for item in analysis["concerns"][:3])
        if not analysis["concerns"]:
            lines.append("- No preference or high-priority delay concern was identified.")
        lines.extend(["", "Context not represented in the schedule data:"])
        lines.extend(f"- {item}" for item in analysis["missing_factors"])
        if modifications:
            lines.extend(["", "Proposed changes:"])
            vessels_by_id = {vessel.id: vessel for vessel in self.scenario["vessels"]}
            for vessel_id, modification in modifications.items():
                lines.append(f"- {vessels_by_id[vessel_id].name}: {modification['reasoning']}")
        return "\n".join(lines)

    def _calculate_decision_confidence(self, analysis: Dict, modifications: Dict) -> float:
        """Return a bounded heuristic confidence indicator for the human review."""
        clarity = (len(analysis["strengths"]) + len(analysis["concerns"])) / 10
        modification_factor = min(0.1, len(modifications) * 0.03)
        experience_factor = min(0.1, self.expert.experience_years / 300)
        return min(0.95, max(0.5, 0.8 + clarity + modification_factor + experience_factor))

    def _update_trust(self, recommendation: AIRecommendation, feedback: HumanFeedback):
        """Apply a bounded illustrative trust update for this interaction."""
        old_trust = self.expert.trust_in_ai
        confidence_alignment = abs(recommendation.confidence_score - 0.8)
        explanation_completeness = min(1.0, len(recommendation.decision_factors) / 5)
        engagement = 1.0 - abs(feedback.acceptance_score - 0.7)
        change = -0.1 * confidence_alignment + 0.1 * explanation_completeness + 0.05 * engagement
        new_trust = max(0.3, min(0.95, old_trust + change))
        self.expert.trust_in_ai = new_trust
        self.trust_evolution.append(
            {"old_trust": old_trust, "new_trust": new_trust, "change": new_trust - old_trust}
        )


print("Human review process defined.")

## 4. Collaborative refinement and diagnostic measures

The collaboration process follows the teaching sequence: generate an AI recommendation, obtain human feedback, apply eligible changes, and calculate diagnostic measures. A reassignment is checked against berth dimensions, tidal entry depth, arrival time, and other vessels' berth-occupancy intervals. The final cost is recalculated with the same cost rule used by the AI. Human satisfaction, AI alignment, collaboration quality, and confidence are summary indicators defined by this notebook, not independently validated measures.

In [ ]:
class SymbioticPartnership:
    """Coordinate AI scheduling, human review, and schedule refinement."""

    def __init__(self, scenario_data: Dict):
        self.scenario = scenario_data
        self.ai_optimizer = ExplainableAIOptimizer(scenario_data)
        self.human_decision_maker = HumanDecisionMaker(
            scenario_data["human_expert"], scenario_data
        )
        self.collaboration_history = []

    def collaborative_decision_making(self) -> CollaborativeSolution:
        """Run the recommendation, review, refinement, and metric sequence."""
        print("Starting the human-AI collaboration workflow.")

        ai_recommendation = self.ai_optimizer.generate_recommendation()
        print("AI recommendation generated.")
        print(f"  Heuristic confidence indicator: {ai_recommendation.confidence_score:.2f}")
        print(f"  Modeled cost: ${ai_recommendation.objective_value:,.0f}")
        print(f"  Vessels assigned: {len(ai_recommendation.vessel_assignments)}")

        human_feedback = self.human_decision_maker.evaluate_ai_recommendation(ai_recommendation)
        print("Human review completed.")
        print(f"  Acceptance indicator: {human_feedback.acceptance_score:.2f}")
        print(f"  Review confidence indicator: {human_feedback.confidence_in_decision:.2f}")
        print(f"  Proposed changes: {len(human_feedback.modifications)}")

        final_solution = self._refine_solution(ai_recommendation, human_feedback)
        metrics = self._calculate_collaboration_metrics(
            ai_recommendation, human_feedback, final_solution
        )
        collaborative_solution = CollaborativeSolution(
            vessel_assignments=final_solution["assignments"],
            start_times=final_solution["start_times"],
            objective_value=final_solution["objective"],
            human_satisfaction_score=metrics["human_satisfaction"],
            ai_alignment_score=metrics["ai_alignment"],
            trust_evolution=metrics["trust_change"],
            collaboration_quality=metrics["quality"],
            final_confidence=metrics["final_confidence"],
        )
        self.collaboration_history.append(
            {
                "ai_recommendation": ai_recommendation,
                "human_feedback": human_feedback,
                "final_solution": collaborative_solution,
            }
        )
        return collaborative_solution

    def _refine_solution(self, ai_rec: AIRecommendation, human_feedback: HumanFeedback) -> Dict:
        """Apply feasible human berth changes and recalculate the objective."""
        assignments = ai_rec.vessel_assignments.copy()
        start_times = ai_rec.start_times.copy()

        for vessel_id, modification in human_feedback.modifications.items():
            if modification["type"] != "berth_reassignment":
                continue

            vessel = next(item for item in self.scenario["vessels"] if item.id == vessel_id)
            berth_id = modification["to"]
            berth = next(item for item in self.scenario["berths"] if item["id"] == berth_id)
            feasible_start = self.ai_optimizer._find_feasible_start_time(
                vessel,
                berth,
                assignments,
                start_times,
                exclude_vessel_id=vessel_id,
            )

            if feasible_start is None:
                print(f"  Change declined for vessel {vessel_id}: no feasible berth time was found.")
                continue

            assignments[vessel_id] = berth_id
            start_times[vessel_id] = feasible_start
            print(f"  Human berth preference applied for vessel {vessel_id} at berth {berth_id}.")

        objective = self._calculate_objective(assignments, start_times)
        return {"assignments": assignments, "start_times": start_times, "objective": objective}

    def _calculate_objective(self, assignments: Dict, start_times: Dict) -> float:
        """Recompute the same preference-adjusted demurrage proxy as the AI."""
        total_cost = 0.0
        vessels_by_id = {vessel.id: vessel for vessel in self.scenario["vessels"]}
        berths_by_id = {berth["id"]: berth for berth in self.scenario["berths"]}
        for vessel_id, berth_id in assignments.items():
            total_cost += self.ai_optimizer._vessel_cost(
                vessels_by_id[vessel_id], berths_by_id[berth_id], start_times[vessel_id]
            )
        return total_cost

    def _calculate_collaboration_metrics(
        self,
        ai_rec: AIRecommendation,
        human_feedback: HumanFeedback,
        final_solution: Dict,
    ) -> Dict:
        """Calculate bounded, illustrative indicators for this interaction."""
        incorporated = sum(
            1
            for vessel_id, modification in human_feedback.modifications.items()
            if modification.get("type") == "berth_reassignment"
            and final_solution["assignments"].get(vessel_id) == modification.get("to")
        )
        human_satisfaction = min(1.0, human_feedback.acceptance_score + 0.1 * incorporated)

        original = ai_rec.vessel_assignments
        final = final_solution["assignments"]
        if original:
            alignment = sum(original.get(vessel_id) == final.get(vessel_id) for vessel_id in original)
            alignment /= len(original)
        else:
            alignment = 0.0

        trust_change = self.human_decision_maker.trust_evolution[-1]["change"]
        quality = (
            human_satisfaction
            + alignment
            + (1.0 - min(1.0, abs(trust_change)))
            + human_feedback.confidence_in_decision
        ) / 4
        confidence = (ai_rec.confidence_score + human_feedback.confidence_in_decision) / 2
        return {
            "human_satisfaction": human_satisfaction,
            "ai_alignment": alignment,
            "trust_change": trust_change,
            "quality": quality,
            "final_confidence": confidence,
        }


print("Collaboration workflow defined.")

## 5. Run the collaboration workflow

The cell below runs one scenario instance and reports elapsed wall-clock time for this small in-memory demonstration. That timing is hardware- and environment-dependent; it is not a benchmark of planning time or a comparison with manual scheduling.

In [ ]:
print("Human-AI berth-allocation demonstration")
print("Scenario: Port of Rotterdam, tidal access windows")
print(f"Human expert: {scenario['human_expert'].name}")
print(f"Vessels: {len(scenario['vessels'])}; berths: {len(scenario['berths'])}")

partnership = SymbioticPartnership(scenario)
workflow_start = time.perf_counter()
collaborative_solution = partnership.collaborative_decision_making()
elapsed_seconds = time.perf_counter() - workflow_start
print(f"\nWorkflow runtime in this execution: {elapsed_seconds:.4f} seconds")

## 6. Inspect the schedule and feedback

The report lists assigned vessels, their start times, modeled delays and costs, and the human-AI indicators. An unassigned vessel is not silently treated as scheduled: the assignment count in the recommendation and the final schedule should be checked when interpreting the results.

In [ ]:
def display_collaboration_results(solution: CollaborativeSolution, partnership: SymbioticPartnership):
    """Print the final schedule and the information behind its indicators."""
    print("\nCOLLABORATIVE SCHEDULE")
    print("=" * 72)
    vessels_by_id = {vessel.id: vessel for vessel in partnership.scenario["vessels"]}
    berths_by_id = {berth["id"]: berth for berth in partnership.scenario["berths"]}

    for vessel_id in sorted(solution.vessel_assignments):
        vessel = vessels_by_id[vessel_id]
        berth_id = solution.vessel_assignments[vessel_id]
        start = solution.start_times[vessel_id]
        delay = max(0.0, start - vessel.arrival_time)
        cost = partnership.ai_optimizer._vessel_cost(vessel, berths_by_id[berth_id], start)
        preference = "yes" if vessel.captain_preference == berth_id else "no or not specified"
        print(
            f"{vessel.name}: {berths_by_id[berth_id]['name']}, "
            f"start {start:.1f} h, delay {delay:.1f} h, "
            f"modeled cost ${cost:,.0f}, preferred berth: {preference}"
        )

    unassigned = set(vessels_by_id) - set(solution.vessel_assignments)
    if unassigned:
        names = ", ".join(vessels_by_id[vessel_id].name for vessel_id in sorted(unassigned))
        print(f"Unassigned vessels: {names}")
    print(f"\nAssigned vessels: {len(solution.vessel_assignments)} / {len(vessels_by_id)}")
    print(f"Total modeled cost: ${solution.objective_value:,.0f}")
    print("\nIllustrative collaboration indicators (0–1 unless noted):")
    print(f"  Human satisfaction indicator: {solution.human_satisfaction_score:.2f}")
    print(f"  AI assignment alignment: {solution.ai_alignment_score:.2f}")
    print(f"  Trust change: {solution.trust_evolution:+.3f}")
    print(f"  Collaboration quality indicator: {solution.collaboration_quality:.2f}")
    print(f"  Combined confidence indicator: {solution.final_confidence:.2f}")

    trust_history = partnership.human_decision_maker.trust_evolution
    if trust_history:
        trust = trust_history[-1]
        print("\nIllustrative trust update:")
        print(f"  Before review: {trust['old_trust']:.3f}")
        print(f"  After review: {trust['new_trust']:.3f}")
        print(f"  Change: {trust['change']:+.3f}")

    record = partnership.collaboration_history[-1]
    feedback = record["human_feedback"]
    recommendation = record["ai_recommendation"]
    print("\nHuman feedback:")
    print(f"  Acceptance indicator: {feedback.acceptance_score:.2f}")
    print(f"  Decision-confidence indicator: {feedback.confidence_in_decision:.2f}")
    print(f"  Proposed changes: {len(feedback.modifications)}")
    print("\nAI recommendation:")
    print(f"  Confidence indicator: {recommendation.confidence_score:.2f}")
    print(f"  Modeled cost: ${recommendation.objective_value:,.0f}")
    print(f"  Decision factors listed: {len(recommendation.decision_factors)}")
    print(f"  Review perspectives listed: {len(recommendation.alternatives)}")


display_collaboration_results(collaborative_solution, partnership)

## 7. Visualize the schedules and indicators

The schedule chart shows berth occupancy intervals and tidal access windows. The assignment comparison distinguishes vessels that remain at the AI-recommended berth from vessels whose final berth differs. A missing assignment is shown separately from a berth number. The trust plot uses interaction order rather than wall-clock timestamps so repeated runs can be interpreted without relying on execution-time metadata.

In [ ]:
def visualize_collaborative_solution(
    solution: CollaborativeSolution,
    partnership: SymbioticPartnership,
):
    """Plot schedules, assignment alignment, indicators, and trust history."""
    scenario_data = partnership.scenario
    vessels_by_id = {vessel.id: vessel for vessel in scenario_data["vessels"]}
    recommendation = partnership.collaboration_history[-1]["ai_recommendation"]
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    colors = ["#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4", "#FFEAA7"]

    ax1.set_title("Final berth schedule")
    ax1.set_xlabel("Time (hours from the scenario origin)")
    ax1.set_ylabel("Berth")
    ax1.set_xlim(12, 27)
    ax1.set_ylim(-0.5, len(scenario_data["berths"]) - 0.5)
    ax1.set_yticks(range(len(scenario_data["berths"])))
    ax1.set_yticklabels([berth["name"] for berth in scenario_data["berths"]])
    ax1.grid(True, axis="x", alpha=0.3)
    for index, window in enumerate(scenario_data["tidal_windows"]):
        ax1.axvspan(
            window["start"],
            window["end"],
            alpha=0.16,
            color="lightblue",
            label="Tidal entry window" if index == 0 else None,
        )
    for vessel_id, berth_id in solution.vessel_assignments.items():
        vessel = vessels_by_id[vessel_id]
        start = solution.start_times[vessel_id]
        rect = patches.Rectangle(
            (start, berth_id - 0.3),
            vessel.processing_time,
            0.6,
            linewidth=1,
            edgecolor="black",
            facecolor=colors[vessel_id % len(colors)],
            alpha=0.85,
        )
        ax1.add_patch(rect)
        ax1.text(
            start + vessel.processing_time / 2,
            berth_id,
            f"V{vessel_id}",
            ha="center",
            va="center",
            fontsize=9,
        )
    ax1.legend(loc="upper left")

    ax2.set_title("AI and final berth assignments")
    vessel_ids = list(range(len(scenario_data["vessels"])))
    positions = np.arange(len(vessel_ids))
    width = 0.36
    ai_values = [recommendation.vessel_assignments.get(item, np.nan) for item in vessel_ids]
    final_values = [solution.vessel_assignments.get(item, np.nan) for item in vessel_ids]
    ax2.bar(positions - width / 2, ai_values, width, label="AI recommendation", color="lightblue")
    ax2.bar(positions + width / 2, final_values, width, label="Final collaborative schedule", color="lightgreen")
    ax2.set_xlabel("Vessel ID")
    ax2.set_ylabel("Berth ID; blank means unassigned")
    ax2.set_xticks(positions)
    ax2.set_xticklabels([f"V{item}" for item in vessel_ids])
    ax2.set_yticks(range(len(scenario_data["berths"])))
    ax2.legend()
    ax2.grid(True, axis="y", alpha=0.3)

    ax3.set_title("Illustrative collaboration indicators")
    metric_names = ["Human\nsatisfaction", "AI\nalignment", "Collaboration\nquality", "Final\nconfidence"]
    metric_values = [
        solution.human_satisfaction_score,
        solution.ai_alignment_score,
        solution.collaboration_quality,
        solution.final_confidence,
    ]
    ax3.bar(metric_names, metric_values, color=["#4ECDC4", "#45B7D1", "#96CEB4", "#FFEAA7"])
    ax3.set_ylim(0, 1)
    ax3.set_ylabel("Indicator value (0–1)")
    ax3.grid(True, axis="y", alpha=0.3)

    ax4.set_title("Illustrative trust history")
    history = partnership.human_decision_maker.trust_evolution
    if history:
        initial_trust = history[0]["old_trust"]
        levels = [initial_trust] + [item["new_trust"] for item in history]
        interactions = range(len(levels))
        ax4.plot(interactions, levels, marker="o", color="purple", label="Trust indicator")
        ax4.axhline(initial_trust, color="gray", linestyle="--", label="Initial trust")
        ax4.set_xticks(list(interactions))
        ax4.set_xlabel("Interaction index")
        ax4.set_ylabel("Trust indicator (0–1)")
        ax4.set_ylim(0, 1)
        ax4.legend()
        ax4.grid(True, alpha=0.3)
    else:
        ax4.text(0.5, 0.5, "No trust interactions recorded", ha="center", va="center")
        ax4.set_axis_off()

    fig.suptitle("Human-AI berth-allocation analysis", fontsize=16)
    fig.tight_layout()
    plt.show()


visualize_collaborative_solution(collaborative_solution, partnership)

## 8. Learning and adaptation analysis

This section interprets the recorded interaction using the notebook's stated thresholds. A single interaction can illustrate how a feedback rule behaves, but it cannot establish a learning trend or demonstrate that a system improves over time. The confidence and quality indicators are heuristic summaries, not calibrated probabilities or validated performance measures.

In [ ]:
def analyze_learning_adaptation(partnership: SymbioticPartnership):
    """Summarize recorded trust, feedback, and recommendation indicators."""
    print("\nLEARNING AND ADAPTATION DIAGNOSTICS")
    print("=" * 72)
    trust_history = partnership.human_decision_maker.trust_evolution
    if trust_history:
        total_change = sum(item["change"] for item in trust_history)
        positive = sum(item["change"] > 0 for item in trust_history)
        negative = sum(item["change"] < 0 for item in trust_history)
        print(f"Recorded trust change: {total_change:+.3f}")
        print(f"Positive updates: {positive}/{len(trust_history)}")
        print(f"Negative updates: {negative}/{len(trust_history)}")
        if len(trust_history) == 1:
            print("Interpretation: one interaction is insufficient to infer a trend.")
        elif total_change > 0.1:
            print("Interpretation: the recorded indicator increased across these interactions.")
        elif total_change < -0.1:
            print("Interpretation: the recorded indicator decreased across these interactions.")
        else:
            print("Interpretation: the recorded indicator changed little overall.")

    if not partnership.collaboration_history:
        print("No collaboration interaction is available to analyze.")
        return

    record = partnership.collaboration_history[-1]
    solution = record["final_solution"]
    feedback = record["human_feedback"]
    recommendation = record["ai_recommendation"]
    print(f"\nCollaboration quality indicator: {solution.collaboration_quality:.2f}")
    print(f"Human acceptance indicator: {feedback.acceptance_score:.2f}")
    print(f"Human decision-confidence indicator: {feedback.confidence_in_decision:.2f}")
    print(f"AI confidence indicator: {recommendation.confidence_score:.2f}")
    print(f"Vessels assigned by AI: {len(recommendation.vessel_assignments)}")
    print(f"Vessels assigned after review: {len(solution.vessel_assignments)}")

    print("\nModeling limits to consider:")
    print("- No multi-interaction data are generated by this single workflow run.")
    print("- Weather, safety margins, and tugboat or pilot scheduling are not quantified.")
    print("- The trust and quality scores are illustrative formulas, not empirical measures.")
    print("- Human feedback changes this schedule; it does not train a persistent AI model.")


analyze_learning_adaptation(partnership)

## 9. Compare the AI and collaborative schedules

This comparison uses the same scenario and modeled cost definition for both schedules. Satisfaction is a preference-based indicator, and operational feasibility checks the modeled berth, depth, tidal-entry, and occupancy rules. The composite score is a transparent, user-chosen summary with fixed weights; it is not an objective function optimized by the scheduling heuristic. One scenario run cannot establish that collaboration is generally superior.

In [ ]:
def calculate_vessel_satisfaction(assignments: Dict[int, int], scenario_data: Dict) -> float:
    """Calculate a preference- and priority-weighted score for assigned vessels."""
    vessels_by_id = {vessel.id: vessel for vessel in scenario_data["vessels"]}
    scores = []
    for vessel_id, berth_id in assignments.items():
        vessel = vessels_by_id[vessel_id]
        if vessel.captain_preference == berth_id:
            preference_score = 1.0
        elif vessel.captain_preference is not None:
            preference_score = 0.3
        else:
            preference_score = 0.7
        scores.append(preference_score * (0.5 + 0.5 * vessel.commercial_priority))
    return float(np.mean(scores)) if scores else 0.0


def assess_operational_feasibility(solution, scenario_data: Dict) -> float:
    """Return the fraction of assigned vessels satisfying modeled feasibility checks."""
    vessels_by_id = {vessel.id: vessel for vessel in scenario_data["vessels"]}
    berths_by_id = {berth["id"]: berth for berth in scenario_data["berths"]}
    optimizer = ExplainableAIOptimizer(scenario_data)
    assignments = solution.vessel_assignments
    start_times = solution.start_times
    if not assignments:
        return 0.0

    feasible_count = 0
    schedules = defaultdict(list)
    for vessel_id, berth_id in assignments.items():
        vessel = vessels_by_id[vessel_id]
        berth = berths_by_id[berth_id]
        start = start_times[vessel_id]
        depth_feasible = any(
            window["start"] <= start <= window["end"]
            and berth["depth_low"]
            + (berth["depth_high"] - berth["depth_low"]) * window["depth_factor"]
            >= vessel.draft
            for window in scenario_data["tidal_windows"]
        )
        feasible = (
            berth["length"] >= vessel.length
            and start >= vessel.arrival_time
            and depth_feasible
        )
        feasible_count += int(feasible)
        schedules[berth_id].append((start, start + vessel.processing_time))

    conflict_count = 0
    for intervals in schedules.values():
        intervals.sort()
        conflict_count += sum(
            intervals[index][0] < intervals[index - 1][1]
            for index in range(1, len(intervals))
        )
    if conflict_count:
        feasible_count = max(0, feasible_count - conflict_count)
    return feasible_count / len(assignments)


def create_comparison_visualization(pure_ai, collaborative, scenario_data: Dict):
    """Plot modeled cost and preference satisfaction for the two schedules."""
    methods = ["AI recommendation", "Human-AI collaboration"]
    costs = [pure_ai.objective_value, collaborative.objective_value]
    satisfaction = [
        calculate_vessel_satisfaction(pure_ai.vessel_assignments, scenario_data),
        calculate_vessel_satisfaction(collaborative.vessel_assignments, scenario_data),
    ]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    bars = ax1.bar(methods, costs, color=["lightblue", "lightgreen"])
    ax1.set_title("Modeled cost comparison")
    ax1.set_ylabel("Preference-adjusted demurrage proxy ($)")
    ax1.tick_params(axis="x", rotation=12)
    ax1.grid(True, axis="y", alpha=0.3)
    for bar, value in zip(bars, costs):
        ax1.annotate(
            f"${value:,.0f}",
            (bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
        )

    bars = ax2.bar(methods, satisfaction, color=["lightcoral", "lightgreen"])
    ax2.set_title("Preference-based satisfaction indicator")
    ax2.set_ylabel("Indicator (0–1)")
    ax2.set_ylim(0, 1)
    ax2.tick_params(axis="x", rotation=12)
    ax2.grid(True, axis="y", alpha=0.3)
    for bar, value in zip(bars, satisfaction):
        ax2.annotate(
            f"{value:.3f}",
            (bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
        )
    fig.tight_layout()
    plt.show()


def compare_approaches(partnership: SymbioticPartnership):
    """Compare the one-run AI and collaborative outcomes using declared indicators."""
    record = partnership.collaboration_history[-1]
    pure_ai = record["ai_recommendation"]
    collaborative = record["final_solution"]
    scenario_data = partnership.scenario

    ai_satisfaction = calculate_vessel_satisfaction(pure_ai.vessel_assignments, scenario_data)
    collaborative_satisfaction = calculate_vessel_satisfaction(
        collaborative.vessel_assignments, scenario_data
    )
    ai_feasibility = assess_operational_feasibility(pure_ai, scenario_data)
    collaborative_feasibility = assess_operational_feasibility(collaborative, scenario_data)

    print("\nAI AND COLLABORATIVE SCHEDULE COMPARISON")
    print("=" * 72)
    print(f"AI modeled cost: ${pure_ai.objective_value:,.0f}")
    print(f"Collaborative modeled cost: ${collaborative.objective_value:,.0f}")
    print(f"Cost difference (collaborative minus AI): ${collaborative.objective_value - pure_ai.objective_value:,.0f}")
    print(f"AI preference satisfaction indicator: {ai_satisfaction:.3f}")
    print(f"Collaborative preference satisfaction indicator: {collaborative_satisfaction:.3f}")
    print(f"AI modeled feasibility fraction: {ai_feasibility:.3f}")
    print(f"Collaborative modeled feasibility fraction: {collaborative_feasibility:.3f}")

    ai_score = (
        (1.0 - min(pure_ai.objective_value / 100000, 1.0)) * 0.4
        + ai_satisfaction * 0.3
        + ai_feasibility * 0.3
    )
    collaborative_score = (
        (1.0 - min(collaborative.objective_value / 100000, 1.0)) * 0.4
        + collaborative_satisfaction * 0.3
        + collaborative_feasibility * 0.3
    )
    print("\nIllustrative composite indicator (cost 0.4, satisfaction 0.3, feasibility 0.3):")
    print(f"  AI recommendation: {ai_score:.3f}")
    print(f"  Human-AI collaboration: {collaborative_score:.3f}")
    print("These weights are diagnostic choices, not learned preferences or an optimized objective.")
    create_comparison_visualization(pure_ai, collaborative, scenario_data)


compare_approaches(partnership)

## 10. Role, strengths, and limitations of Tier 7

### What the human-AI workflow contributes
- **Contextual review:** an expert can identify operational information that is not present in the structured input data.
- **Explainability:** the recommendation reports the modeled constraints, priority order, and cost proxy.
- **Human authority:** a proposed change is considered explicitly and checked against the modeled feasibility rules.
- **Feedback recording:** acceptance, proposed changes, and a simple trust indicator are retained for inspection.

### Trade-offs and limitations
- The heuristic is priority-ordered and greedy; it does not guarantee a globally optimal schedule.
- The scenario does not provide enough information to enforce tugboat, pilot, weather, channel-traffic, or safety-margin constraints.
- A human review can improve contextual fit, but it can also increase cost or create infeasibility if checks are incomplete. The code checks only the constraints explicitly represented in the model.
- The notebook simulates a single review cycle. It does not implement an interactive user interface, persistent learning, or a measured comparison with manual planning.
- The satisfaction, trust, confidence, and composite scores are illustrative indicators whose definitions and limitations should be considered before any operational use.

### When this approach is useful
A human-in-the-loop workflow is worth considering when decisions involve high stakes, unstructured context, changing operating conditions, or stakeholder relationships that are not adequately represented by the optimization inputs. Before deployment, the model would need validated operational data, explicit safety and resource constraints, user testing, and a documented process for reviewing infeasible or incomplete recommendations.

### Interpretation of this scenario
Use the printed schedule and checks to inspect what the heuristic and review rules do on the supplied data. A difference between the AI and collaborative schedules demonstrates the effect of the specified human preference rule in this example; it does not establish a general improvement in cost, safety, robustness, or user adoption.